# Object-Oriented API: Advanced Control

The OO API provides more control and features:

1. Full configuration control
2. State management and history tracking
3. Serialization/deserialization
4. Multiple fit operations
5. Summary generation

## Setup

In [1]:
import sys
from pathlib import Path

# Add src to path for importing pso_segmentation
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from pso_segmentation import (
    OptimizerConfig,
    SegmentationOptimizer,
    example_fitness_r2_with_all_constraints,
    select_n_segments,
)

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## Generate Sample Data

In [2]:
n_samples = 2000
scores = np.random.beta(a=2, b=5, size=n_samples)
labels = (np.random.rand(n_samples) < scores).astype(int)

print(f"Dataset: {n_samples} customers")
print(f"Default rate: {labels.mean():.1%}")

Dataset: 2000 customers
Default rate: 28.4%


## 1. Basic OO Workflow

In [3]:
# Step 1: Create configuration
config = OptimizerConfig(n_segments=4, pop_size=80, max_iter=300, seed=42)

# Step 2: Create optimizer
optimizer = SegmentationOptimizer(config)


# Step 3: Define fitness function
def fitness_func(cuts):
    return example_fitness_r2_with_all_constraints(cuts, scores, labels)


# Step 4: Fit
optimizer.fit(scores, labels, fitness_func)

# Step 5: Get results
result = optimizer.get_metrics()

print("Optimization complete!")
print(f"Best R²: {result.r2:.4f}")

Optimization complete!
Best R²: 0.1345


## 2. Access Detailed Information

In [4]:
# Get cuts
cuts = optimizer.get_cuts()
print(f"Segment cuts: {np.round(cuts, 3)}")

# Generate summary
print("\n" + "=" * 60)
print("OPTIMIZER SUMMARY")
print("=" * 60)
print(optimizer.summary())

Segment cuts: [0.187 0.315 0.453]

OPTIMIZER SUMMARY
SEGMENTATION OPTIMIZER RESULTS
R² (Variance Explained): 0.1345
Number of Segments: 4

Cut Boundaries:
  Cut 1: 0.186670
  Cut 2: 0.315389
  Cut 3: 0.453364

Segment Statistics:
  Segment 0: PD=12.06%, Size=29.85% (n=597)
  Segment 1: PD=21.92%, Size=29.65% (n=593)
  Segment 2: PD=34.47%, Size=23.50% (n=470)
  Segment 3: PD=60.29%, Size=17.00% (n=340)

Constraint Validation:
  Valid: True


## 3. Configuration Options

In [5]:
# Show all available config options
print("OptimizerConfig Parameters:")
print("-" * 60)

config_dict = {
    "n_segments": "Number of segments to create (default: 5)",
    "min_segment_size": "Minimum segment proportion (default: 0.05)",
    "max_segment_size": "Maximum segment proportion (default: 0.30)",
    "enforce_monotonic": "Force monotonic default rates (default: False)",
    "pop_size": "PSO population size (default: 50)",
    "max_iter": "Max PSO iterations (default: 500)",
    "w": "Inertia weight (default: 0.7)",
    "c1": "Cognitive parameter (default: 1.5)",
    "c2": "Social parameter (default: 1.5)",
    "track_history": "Track optimization history (default: True)",
    "seed": "Random seed for reproducibility (default: None)",
}

for param, desc in config_dict.items():
    print(f"  {param:20s}: {desc}")

OptimizerConfig Parameters:
------------------------------------------------------------
  n_segments          : Number of segments to create (default: 5)
  min_segment_size    : Minimum segment proportion (default: 0.05)
  max_segment_size    : Maximum segment proportion (default: 0.30)
  enforce_monotonic   : Force monotonic default rates (default: False)
  pop_size            : PSO population size (default: 50)
  max_iter            : Max PSO iterations (default: 500)
  w                   : Inertia weight (default: 0.7)
  c1                  : Cognitive parameter (default: 1.5)
  c2                  : Social parameter (default: 1.5)
  track_history       : Track optimization history (default: True)
  seed                : Random seed for reproducibility (default: None)


## 4. History Tracking

When `track_history=True`, monitor PSO convergence:

In [6]:
# Create optimizer with history tracking
config_hist = OptimizerConfig(
    n_segments=4,
    pop_size=60,
    max_iter=500,
    track_history=True,  # Enable history tracking
    seed=42,
)

optimizer_hist = SegmentationOptimizer(config_hist)
optimizer_hist.fit(scores, labels, fitness_func)

# Get optimization history
history = optimizer_hist.get_history()
best_fitness_history = [h["best_fitness"] for h in history]

print("Optimization history tracked:")
print(f"  Total iterations: {len(history)}")
print(f"  Initial fitness: {best_fitness_history[0]:.4f}")
print(f"  Final fitness: {best_fitness_history[-1]:.4f}")
print(f"  Improvement: {(best_fitness_history[-1] - best_fitness_history[0]):.4f}")
print(f"\nFirst 10 iterations: {np.round(best_fitness_history[:10], 4)}")
print(f"Last 10 iterations: {np.round(best_fitness_history[-10:], 4)}")

Optimization history tracked:
  Total iterations: 500
  Initial fitness: 0.1030
  Final fitness: 0.1313
  Improvement: 0.0283

First 10 iterations: [0.103  0.1031 0.1031 0.1031 0.1247 0.1247 0.1247 0.1247 0.1247 0.1247]
Last 10 iterations: [0.1313 0.1313 0.1313 0.1313 0.1313 0.1313 0.1313 0.1313 0.1313 0.1313]


## 5. Serialization

Save and load optimizer state:

In [7]:
from pso_segmentation import load_optimizer_state, save_optimizer_state

# Create portable output directory
artifact_dir = Path("./artifacts")
artifact_dir.mkdir(exist_ok=True)

# Save to JSON (results + cuts)
json_path = artifact_dir / "segmentation.json"
optimizer.to_json(str(json_path))
print(f"Saved to JSON: {json_path}")

# Load from JSON
optimizer_loaded = SegmentationOptimizer.from_json(str(json_path))
result_loaded = optimizer_loaded.get_metrics()
print(f"Loaded from JSON - R²: {result_loaded.r2:.4f}")

# Save to pickle (full state)
pickle_path = artifact_dir / "optimizer.pkl"
save_optimizer_state(optimizer, str(pickle_path))
print(f"\nSaved to pickle: {pickle_path}")

# Load from pickle
optimizer_pickled = load_optimizer_state(str(pickle_path))
result_pickled = optimizer_pickled.get_metrics()
print(f"Loaded from pickle - R²: {result_pickled.r2:.4f}")

Saved to JSON: artifacts\segmentation.json
Loaded from JSON - R²: 0.1345

Saved to pickle: artifacts\optimizer.pkl
Loaded from pickle - R²: 0.1345


## 6. Different PSO Parameters

Compare different PSO configurations:

In [8]:
# Test different PSO parameter settings
configs = [
    {"name": "Conservative", "w": 0.4, "c1": 1.5, "c2": 1.5, "pop": 30},
    {"name": "Balanced", "w": 0.7, "c1": 1.5, "c2": 1.5, "pop": 50},
    {"name": "Exploratory", "w": 0.9, "c1": 2.0, "c2": 2.0, "pop": 100},
]

results_comparison = []

for cfg in configs:
    opt_config = OptimizerConfig(
        n_segments=4,
        w=cfg["w"],
        c1=cfg["c1"],
        c2=cfg["c2"],
        pop_size=cfg["pop"],
        max_iter=200,
        seed=42,
    )

    opt = SegmentationOptimizer(opt_config)
    opt.fit(scores, labels, fitness_func)
    res = opt.get_metrics()

    results_comparison.append(
        {
            "Config": cfg["name"],
            "w": cfg["w"],
            "pop_size": cfg["pop"],
            "R²": res.r2,
            "H_inter": res.h_inter,
            "H_intra": res.h_intra,
        }
    )

df_comparison = pd.DataFrame(results_comparison)
print("PSO Parameter Comparison:")
print(df_comparison.to_string(index=False))

PSO Parameter Comparison:
      Config   w  pop_size       R²   H_inter    H_intra
Conservative 0.4        30 0.140257 57.101184 350.018316
    Balanced 0.7        50 0.131303 53.455949 353.663551
 Exploratory 0.9       100 0.130293 53.044924 354.074576


## 7. Multiple Optimizations

Run multiple optimizations and select the best:

In [9]:
# Run multiple optimizations with different seeds
results = []

for seed in [42, 43, 44, 45]:
    config = OptimizerConfig(n_segments=4, max_iter=300, seed=seed)

    opt = SegmentationOptimizer(config)
    opt.fit(scores, labels, fitness_func)
    res = opt.get_metrics()

    results.append({"Seed": seed, "R²": res.r2, "Optimizer": opt, "Result": res})

df_results = pd.DataFrame(results)
print("Multiple Runs Comparison:")
print(df_results[["Seed", "R²"]].to_string(index=False))

# Select best
best_idx = df_results["R²"].idxmax()
best_optimizer = df_results.loc[best_idx, "Optimizer"]
best_result = df_results.loc[best_idx, "Result"]

print(f"\nBest run (Seed {df_results.loc[best_idx, 'Seed']}): R² = {best_result.r2:.4f}")

Multiple Runs Comparison:
 Seed       R²
   42 0.131303
   43 0.134465
   44 0.131303
   45 0.131303

Best run (Seed 43): R² = 0.1345


## 8. Selecting the Number of Segments

`select_n_segments` runs one optimizer per candidate segment count and optional constraint-parameter combination, validates each candidate with the configured constraints, and keeps the best valid solution. This is useful when the number of risk tiers is a modeling choice rather than a fixed input.

In [10]:
def constrained_objective_factory(scores, labels, _n_segments, params):
    """Build an objective for each candidate number of segments."""

    def objective(cuts):
        return example_fitness_r2_with_all_constraints(
            cuts,
            scores,
            labels,
            monotonic_weight=params["monotonic_weight"],
            balance_weight=params["balance_weight"],
        )

    return objective


selection_config = OptimizerConfig(
    pop_size=50,
    max_iter=150,
    min_segment_size=0.05,
    max_segment_size=0.40,
    enforce_monotonic=True,
    seed=42,
)

selection = select_n_segments(
    scores,
    labels,
    segment_range=(3, 6),
    objective_factory=constrained_objective_factory,
    base_config=selection_config,
    param_grid={
        "monotonic_weight": [0.2, 0.4],
        "balance_weight": [0.1, 0.2],
    },
    selection_metric="r2",
)

candidate_rows = [
    {
        "Segments": candidate.n_segments,
        "Mono weight": candidate.params["monotonic_weight"],
        "Balance weight": candidate.params["balance_weight"],
        "R²": candidate.metrics.r2,
        "Gini": candidate.metrics.gini,
        "KS": candidate.metrics.ks,
        "Valid": candidate.valid,
        "Validation": candidate.validation_message,
    }
    for candidate in selection.candidates
]

df_candidates = pd.DataFrame(candidate_rows)
print("Segment count candidates:")
print(df_candidates.round(4).to_string(index=False))

print(f"\nSelected number of segments: {selection.best_n_segments}")
print(f"Selected R²: {selection.best_metrics.r2:.4f}")

Segment count candidates:
 Segments  Mono weight  Balance weight     R²   Gini     KS  Valid                                   Validation
        3          0.2             0.1 0.1358 0.3755 0.3282  False Maximum segment proportion (0.4475) > 0.4000
        3          0.2             0.2 0.1358 0.3755 0.3282  False Maximum segment proportion (0.4475) > 0.4000
        3          0.4             0.1 0.1358 0.3755 0.3282  False Maximum segment proportion (0.4475) > 0.4000
        3          0.4             0.2 0.1358 0.3755 0.3282  False Maximum segment proportion (0.4475) > 0.4000
        4          0.2             0.1 0.1345 0.2611 0.3354   True                        Segmentation is valid
        4          0.2             0.2 0.1313 0.2590 0.3282   True                        Segmentation is valid
        4          0.4             0.1 0.1345 0.2611 0.3354   True                        Segmentation is valid
        4          0.4             0.2 0.1313 0.2590 0.3282   True            

## 9. Production Workflow

Complete end-to-end production workflow:

In [11]:
print("PRODUCTION WORKFLOW")
print("=" * 60)

# 1. Configure
print("\n1. Configuring optimizer...")
config = OptimizerConfig(
    n_segments=5, pop_size=100, max_iter=500, enforce_monotonic=False, track_history=True, seed=42
)
print(f"   Configured for {config.n_segments} segments")

# 2. Create optimizer
print("\n2. Creating optimizer...")
optimizer = SegmentationOptimizer(config)
print("   Optimizer created")

# 3. Fit
print("\n3. Fitting optimizer...")
optimizer.fit(scores, labels, fitness_func)
print("   Optimization complete")

# 4. Get result
print("\n4. Retrieving results...")
result = optimizer.get_metrics()
print(f"   R² = {result.r2:.4f}")

# 5. Validate
print("\n5. Validating results...")
pd_by_seg = result.pd_by_segment
is_valid = all(0 <= pd <= 1 for pd in pd_by_seg)
print(f"   Valid: {is_valid}")
print(f"   PD range: [{pd_by_seg.min():.3f}, {pd_by_seg.max():.3f}]")

# 6. Save
print("\n6. Saving optimizer state...")
artifact_dir = Path("./artifacts")
artifact_dir.mkdir(exist_ok=True)
prod_pickle_path = artifact_dir / "prod_optimizer.pkl"
save_optimizer_state(optimizer, str(prod_pickle_path))
print(f"   Saved to {prod_pickle_path}")

# 7. Generate summary
print("\n7. Summary:")
print(optimizer.summary())

print("\n" + "=" * 60)
print("WORKFLOW COMPLETE")

PRODUCTION WORKFLOW

1. Configuring optimizer...
   Configured for 5 segments

2. Creating optimizer...
   Optimizer created

3. Fitting optimizer...
   Optimization complete

4. Retrieving results...
   R² = 0.1438

5. Validating results...
   Valid: True
   PD range: [0.085, 0.657]

6. Saving optimizer state...
   Saved to artifacts\prod_optimizer.pkl

7. Summary:
SEGMENTATION OPTIMIZER RESULTS
R² (Variance Explained): 0.1438
Number of Segments: 5

Cut Boundaries:
  Cut 1: 0.131723
  Cut 2: 0.248075
  Cut 3: 0.377091
  Cut 4: 0.521095

Segment Statistics:
  Segment 0: PD=8.48%, Size=17.10% (n=342)
  Segment 1: PD=16.64%, Size=27.65% (n=553)
  Segment 2: PD=28.44%, Size=26.90% (n=538)
  Segment 3: PD=44.72%, Size=18.45% (n=369)
  Segment 4: PD=65.66%, Size=9.90% (n=198)

Constraint Validation:
  Valid: True

WORKFLOW COMPLETE


## Key Takeaways

✅ **OO API Benefits:**
- Full configuration control
- State management and persistence
- History tracking for convergence analysis
- Multiple operations on same instance
- Professional summary generation

✅ **Configuration Tuning:**
- Adjust w, c1, c2 for PSO behavior
- Control pop_size and max_iter
- Enable history tracking for analysis
- Set seed for reproducibility

✅ **Serialization:**
- JSON for portability
- Pickle for full state preservation

## Next Steps

👉 **03_custom_fitness.ipynb** - Build custom fitness functions

👉 **04_business_use_case.ipynb** - Real-world application